In [ ]:
# ============================================================
# IMAGE SHA-256 HASHING MODULE
# Capstone: Tampered Image Detection
# ============================================================

import hashlib
import csv
import json
import os
from datetime import datetime, timezone

from google.colab import drive, files

In [ ]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

print("Mounting Google Drive...")
drive.mount("/content/drive")

print("Google Drive mounted successfully.")
print()


In [ ]:
# ============================================================
# 2. ORIGINAL IMAGE FOLDER
# ============================================================

ORIGINAL_FOLDER = (
    "/content/drive/MyDrive/"
    "capstone_dataset creation/"
    "capstone_dataset creation/"
    "original"
)

CSV_FILE = "image_hashes.csv"
JSON_FILE = "image_records.json"

In [ ]:
# ============================================================
# 3. CHECK FOLDER
# ============================================================

if not os.path.exists(ORIGINAL_FOLDER):
    raise FileNotFoundError(
        f"Original folder not found:\n{ORIGINAL_FOLDER}\n\n"
        "Please check the Google Drive folder path."
    )

print("Original folder found:")
print(ORIGINAL_FOLDER)
print()

In [ ]:
# ============================================================
# 4. SUPPORTED IMAGE EXTENSIONS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}


In [ ]:
# ============================================================
# 5. SHA-256 HASH FUNCTION
# ============================================================
# The file is read in binary mode.
#
# We DO NOT:
# - resize/modify/convert/normalize the image
# - open/decode the image
#
# Therefore, SHA-256 is calculated from the EXACT ORIGINAL FILE BYTES.
#
# ============================================================

def calculate_sha256(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:

        while True:

            data = f.read(1024 * 1024)  # Read 1 MB at a time

            if not data:
                break

            sha256.update(data)

    return sha256.hexdigest()


In [ ]:
# ============================================================
# 6. GET IMAGE FILES
# ============================================================

def get_image_files(folder_path):

    image_files = []

    for filename in sorted(os.listdir(folder_path)):

        file_path = os.path.join(folder_path, filename)

        if not os.path.isfile(file_path):
            continue

        extension = os.path.splitext(filename)[1].lower()

        if extension in IMAGE_EXTENSIONS:
            image_files.append(file_path)

    return image_files

In [ ]:
# ============================================================
# 7. LOAD EXISTING RECORDS
# ============================================================

def load_existing_records():

    if not os.path.exists(JSON_FILE):
        return []

    try:

        with open(
            JSON_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            records = json.load(f)

            if isinstance(records, list):
                return records

            return []

    except (json.JSONDecodeError, OSError):

        print("Could not read existing JSON records.")
        print("Starting with an empty record list.")
        print()

        return []

In [ ]:
# ============================================================
# 8. GET NEXT IMAGE ID
# ============================================================

def get_next_image_id(records):

    highest_id = 0

    for record in records:

        image_id = record.get("image_id", "")

        if image_id.startswith("IMG_"):

            try:

                number = int(image_id.split("_")[1])

                highest_id = max(
                    highest_id,
                    number
                )

            except (ValueError, IndexError):

                continue

    return highest_id + 1

In [ ]:
# ============================================================
# 9. SAVE CSV
# ============================================================

def save_csv(records):

    fieldnames = [
        "image_id",
        "filename",
        "sha256",
        "timestamp"
    ]

    with open(
        CSV_FILE,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        writer.writeheader()

        for record in records:
            writer.writerow(record)

In [ ]:
# ============================================================
# 10. SAVE JSON
# ============================================================

def save_json(records):

    with open(
        JSON_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            records,
            f,
            indent=4
        )

In [ ]:
# ============================================================
C11. PROCESS ORIGINAL IMAGES
# ============================================================
#
# This function:
#
# 1. Loads old records.
# 2. Reads every current image.
# 3. Calculates its current SHA-256.
# 4. Compares it with the stored hash.
#
# Cases:
#
# - New filename
#       → create new record
#
# - Existing filename + same hash
#       → keep old record unchanged
#
# - Existing filename + different hash
#       → update hash and timestamp
#
# ============================================================

def process_original_images():

    existing_records = load_existing_records()

    image_files = get_image_files(ORIGINAL_FOLDER)

    if not image_files:

        print("No image files found in the original folder.")
        return existing_records, False

    print(f"Images currently in folder : {len(image_files)}")
    print(f"Existing hash records      : {len(existing_records)}")
    print()

    # --------------------------------------------------------
    # Create lookup table using  filename
    # --------------------------------------------------------

    record_lookup = {
        record["filename"]: record
        for record in existing_records
    }

    next_id_number = get_next_image_id(existing_records)

    new_count = 0
    unchanged_count = 0
    changed_count = 0

    records_changed = False

    # --------------------------------------------------------
    # Process every current image
    # --------------------------------------------------------

    for file_path in image_files:

        filename = os.path.basename(file_path)

        # ----------------------------------------------------
        # Calculate CURRENT SHA-256
        #
        # This happens every run intentionally.
        # Therefore, even if someone replaces a file while
        # keeping the same filename, the change is detected.
        # ----------------------------------------------------

        current_hash = calculate_sha256(file_path)

        # ----------------------------------------------------
        # CASE 1: New filename
        # ----------------------------------------------------

        if filename not in record_lookup:

            image_id = f"IMG_{next_id_number:04d}"

            timestamp = datetime.now(
                timezone.utc
            ).isoformat()

            record = {
                "image_id": image_id,
                "filename": filename,
                "sha256": current_hash,
                "timestamp": timestamp
            }

            existing_records.append(record)

            record_lookup[filename] = record

            next_id_number += 1
            new_count += 1
            records_changed = True

            print("=" * 75)
            print("NEW IMAGE")
            print(f"Image ID   : {image_id}")
            print(f"Filename   : {filename}")
            print(f"SHA-256    : {current_hash}")
            print(f"Timestamp  : {timestamp}")
            print("=" * 75)
            print()

        # ----------------------------------------------------
        # CASE 2: Existing filename
        # ----------------------------------------------------

        else:

            record = record_lookup[filename]

            stored_hash = record["sha256"]

            # ------------------------------------------------
            # Same file contents
            # ------------------------------------------------

            if current_hash == stored_hash:

                unchanged_count += 1

                print(
                    f"UNCHANGED  : {filename} "
                    f"→ existing hash retained"
                )

            # ------------------------------------------------
            # Same filename but different file contents
            # ------------------------------------------------

            else:

                old_hash = record["sha256"]

                new_timestamp = datetime.now(
                    timezone.utc
                ).isoformat()

                record["sha256"] = current_hash
                record["timestamp"] = new_timestamp

                changed_count += 1
                records_changed = True

                print("=" * 75)
                print("IMAGE CHANGED")
                print(f"Image ID   : {record['image_id']}")
                print(f"Filename   : {filename}")
                print(f"Old Hash   : {old_hash}")
                print(f"New Hash   : {current_hash}")
                print(f"Timestamp  : {new_timestamp}")
                print("=" * 75)
                print()

    # --------------------------------------------------------
    # Save only if something changed
    # --------------------------------------------------------

    if records_changed:

        save_csv(existing_records)
        save_json(existing_records)

        print("=" * 75)
        print("HASHING RECORDS UPDATED")
        print("=" * 75)

        print(f"New images       : {new_count}")
        print(f"Changed images   : {changed_count}")
        print(f"Unchanged images : {unchanged_count}")
        print(f"Total records    : {len(existing_records)}")
        print()

        print(f"CSV file  : {CSV_FILE}")
        print(f"JSON file : {JSON_FILE}")
        print()

    else:

        print()
        print("=" * 75)
        print("NO CHANGES DETECTED")
        print("=" * 75)

        print(f"New images       : {new_count}")
        print(f"Changed images   : {changed_count}")
        print(f"Unchanged images : {unchanged_count}")
        print()

        print("Existing CSV and JSON files remain unchanged.")
        print()

    return existing_records, records_changed


In [ ]:
# ============================================================
# 12. VERIFICATION FUNCTION
# ============================================================
# This allows us to select an image and compare its current
# SHA-256 hash against ALL stored hash records.
# Filename is NOT used for matching because Google Colab may
# rename a repeated upload such as:
# image2.jpg
# image2 (1).jpg
# The SHA-256 hash is used for the actual verification.
# ============================================================

def verify_image():

    print("Select an image to verify.")
    print()

    uploaded_files = files.upload()

    if not uploaded_files:

        print("No file uploaded.")
        return

    if not os.path.exists(JSON_FILE):

        print("No stored hash records found.")
        return

    try:

        with open(
            JSON_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            existing_records = json.load(f)

    except (json.JSONDecodeError, OSError):

        print("Could not read stored hash records.")
        return

    for filename, file_bytes in uploaded_files.items():

        # ----------------------------------------------------
        # Calculate hash from EXACT uploaded bytes
        # ----------------------------------------------------

        current_hash = hashlib.sha256(
            file_bytes
        ).hexdigest()

        print()
        print("=" * 75)
        print(f"Verifying       : {filename}")
        print(f"Current SHA-256 : {current_hash}")
        print("=" * 75)

        # ----------------------------------------------------
        # Compare uploaded hash with ALL stored hashes
        # ----------------------------------------------------

        matching_records = [

            record
            for record in existing_records
            if record["sha256"] == current_hash

        ]


        if matching_records:

            matched_record = matching_records[0]

            print()
            print("Image integrity verified - Hash matches")
            print()
            print(f"Matched Image ID : {matched_record['image_id']}")
            print(f"Original Filename: {matched_record['filename']}")
            print()

        # ----------------------------------------------------
        # Hash did not match
        # ----------------------------------------------------

        else:

            print()
            print("Image has changed - Hash mismatch")
            print()

In [ ]:
# ============================================================
# 13. RUN HASHING
# ============================================================

records, records_changed = process_original_images()

In [ ]:
# ============================================================
# 14. DOWNLOAD OUTPUT FILES
# ============================================================

if records_changed:

    print("Changes were detected.")
    print("Preparing updated output files for download...")
    print()

    files.download(CSV_FILE)
    files.download(JSON_FILE)

    print()
    print("Updated CSV and JSON files have been downloaded.")

else:

    print("No changes detected in the original images.")
    print()

    download_choice = input(
        "Do you want to download the current CSV and JSON files? (y/n): "
    ).strip().lower()

    if download_choice == "y":

        files.download(CSV_FILE)
        files.download(JSON_FILE)

        print()
        print("Current CSV and JSON files have been downloaded.")

    else:

        print()
        print("Download skipped.")

In [ ]:
# VERIFICATION (ONLY IF NEEDED)

verify_image()

# -----------------------------------
# END OF HASHING MODULE
# -----------------------------------